[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_19/MFM_ch19_seminar/MFM_ch19_seminar.ipynb)

# MFM_ch19_seminar

Seminar 19 (project workshop): Hill tail indices and a heteroskedasticity-robust portmanteau for BET and Bitcoin, GARCH-t and GJR-GARCH-t for BET with Wald and profile-likelihood intervals for persistence, VaR and ES backtests for Bitcoin with DQ, FZ0 and the model confidence set, the calendar-join pitfall for Bitcoin and SPY, and a reference analysis of BET tail risk around the 2020 FTSE reclassification (Hill indices, difference-in-differences with WIG20, placebo dates, sup-Wald break test).

*Modelling Financial Markets (MFM), Daniel Traian Pele, Bucharest University of Economic Studies*

In [ ]:
# Colab: !pip install arch
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, eticheta, tip, data de start)
MARKETS = {
    'sp500':  ('GSPC.INDX',    'S&P 500',                  'index',  '2000-01-01'),
    'bet':    ('BET',          'BET',                      'index',  '2000-01-01'),
    'btc':    ('BTC-USD.CC',   'Bitcoin',                  'crypto', '2014-09-17'),
    'eurron': ('REF:EUR',      'EUR/RON (reference rate)', 'fx',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX', 'Gold (XAU/USD)',           'fx',     '2000-01-01'),
}
# active pentru portofolii (join pe preturi in zilele comune)
ASSETS = {
    'SPY': ('SPY.US', 'adjusted_close'), 'TLT': ('TLT.US', 'adjusted_close'), 'GLD': ('GLD.US', 'adjusted_close'),
    'BTC': ('BTC-USD.CC', 'close'),
    'TLV': ('TLV.RO', 'adjusted_close'), 'SNP': ('SNP.RO', 'adjusted_close'), 'BRD': ('BRD.RO', 'adjusted_close'),
    'TGN': ('TGN.RO', 'adjusted_close'), 'SNG': ('SNG.RO', 'adjusted_close'), 'SNN': ('SNN.RO', 'adjusted_close'),
    'EL': ('EL.RO', 'adjusted_close'), 'FP': ('FP.RO', 'adjusted_close'), 'TEL': ('TEL.RO', 'adjusted_close'),
    'WIG20': ('WIG20.INDX', 'close'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Pretul zilnic, curatat dupa conventiile capitolului."""
    symbol, _, kind, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
    if kind == 'index':
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Randamente log pe calendarul propriu al seriei."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def periods_per_year(r):
    """Frecventa reala: numarul mediu de observatii pe an calendaristic."""
    return len(r) / ((r.index[-1] - r.index[0]).days / 365.25)


def asset_price(key, end=END):
    symbol, col = ASSETS[key]
    s = read_market(symbol)[col].loc[:end]
    s = s[s > 0].dropna()
    if key != 'BTC':
        s = s[s.index.dayofweek < 5]
    return s.rename(key)


def joint_returns(keys, start=None, end=END):
    """Randamente log simple pentru mai multe active: join pe PRETURI in zilele comune, apoi randamente."""
    p = pd.concat([asset_price(k, end) for k in keys], axis=1, join='inner').dropna()
    if start:
        p = p.loc[start:]
    return np.log(p).diff().dropna()

## Inference functions

In [ ]:
# =============================================================================
# 1. INDICELE DE COADA HILL
# =============================================================================
def hill(losses, k):
    """Estimatorul Hill al indicelui de coada din cele mai mari k pierderi (pozitive)."""
    x = np.sort(np.asarray(losses)[np.asarray(losses) > 0])[::-1]
    return 1.0 / np.mean(np.log(x[:k]) - np.log(x[k]))


def hill_k(r, share=0.05):
    """k = 5% din zilele cu pierdere (aceeasi regula ca in Capitolul 16)."""
    return int(round(share * np.sum(-np.asarray(r) > 0)))


def hill_ci(r, share=0.05, B=999, block=20, seed=0):
    """Hill pe pierderi, CI asimptotic i.i.d. alpha(1 +/- 1.96/sqrt(k)) si CI bootstrap pe blocuri mobile."""
    L = -np.asarray(r)
    k = hill_k(r, share)
    a = hill(L, k)
    rng = np.random.default_rng(seed)
    T = len(L)
    nb = int(np.ceil(T / block))
    bs = np.array([hill(L[(rng.integers(0, T - block, nb)[:, None] + np.arange(block)).ravel()[:T]], k)
                   for _ in range(B)])
    return dict(alpha=a, k=k, se_iid=a / np.sqrt(k), lo_iid=a - 1.96 * a / np.sqrt(k), hi_iid=a + 1.96 * a / np.sqrt(k),
                lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), se_boot=bs.std(ddof=1), draws=bs)


# =============================================================================
# 2. GARCH(1,1)-t: CI PRIN VEROSIMILITATEA PROFIL PENTRU PERSISTENTA
# =============================================================================
def garch_t_nll(theta, r):
    """-log L pentru GARCH(1,1) cu inovatii Student-t standardizate; theta = (mu, omega, alpha, beta, nu)."""
    mu, om, a, b, nu = theta
    if om <= 0 or a < 0 or b < 0 or a + b >= 1 or nu <= 2.05:
        return 1e10
    e = r - mu
    T = len(e)
    s2 = np.empty(T)
    s2[0] = np.var(e)
    for t in range(1, T):
        s2[t] = om + a * e[t - 1] ** 2 + b * s2[t - 1]
    c = (stats.t.logpdf(e / np.sqrt(s2 * (nu - 2) / nu), nu) - 0.5 * np.log(s2 * (nu - 2) / nu))
    return -np.sum(c)


def profile_pers(r, grid, x0):
    """Maximul verosimilitatii cu persistenta fixata p = alpha + beta (reparametrizare alpha = p*s, beta = p*(1-s))."""
    r = np.asarray(r)
    out = []
    for p in grid:
        def f(th):
            mu, om, s, nu = th
            if not 0 < s < 1:
                return 1e10
            return garch_t_nll((mu, om, p * s, p * (1 - s), nu), r)
        best = optimize.minimize(f, x0, method='Nelder-Mead', options={'xatol': 1e-6, 'fatol': 1e-3, 'maxiter': 4000})
        out.append(-best.fun)
        x0 = best.x
    return np.array(out)


def profile_ci(r, res):
    """CI 95% {p < 1 : 2(l_max - l(p)) <= 3.84}; l_max din estimarea nerestrictionata."""
    p = res.params
    pers = p['alpha[1]'] + p['beta[1]']
    lmax = -garch_t_nll((p['mu'], p['omega'], p['alpha[1]'], p['beta[1]'], p['nu']), np.asarray(r))
    x0 = np.array([p['mu'], p['omega'], p['alpha[1]'] / pers, p['nu']])
    grid = np.unique(np.concatenate([np.linspace(0.960, 0.998, 39), [0.999, 0.9995, 0.9999, pers]]))
    lp = profile_pers(r, grid, x0)
    lr = 2 * (max(lmax, lp.max()) - lp)
    ok = grid[lr <= stats.chi2.ppf(0.95, 1)]
    return dict(pers=pers, lmax=lmax, lo=ok.min(), hi=ok.max(), grid=list(grid), lr=list(lr),
                hl_lo=np.log(0.5) / np.log(ok.min()), hl_hi=np.log(0.5) / np.log(ok.max()),
                hi_at_edge=bool(ok.max() >= grid.max()))


# =============================================================================
# 3. TESTE PENTRU PROGNOZELE DE RISC
# =============================================================================
def dq_test(L, var, a=0.01, lags=4):
    """Dynamic Quantile (Engle & Manganelli, 2004): Hit_t - a pe constanta, lags depasiri si VaR_t; DQ ~ chi2(lags+2)."""
    hit = (np.asarray(L) > np.asarray(var)).astype(float) - a
    X = [np.ones(len(hit))] + [np.roll(hit, l) for l in range(1, lags + 1)] + [np.asarray(var)]
    X = np.column_stack(X)[lags:]
    y = hit[lags:]
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    dq = b @ X.T @ X @ b / (a * (1 - a))
    return dict(DQ=dq, df=X.shape[1], p=stats.chi2.sf(dq, X.shape[1]))


def fz0(r, var, es, a):
    """Scorul FZ0 (Patton, Ziegel & Chen, 2019) pentru v = -VaR, e = -ES (cuantile negative), randamentul r."""
    v, e, y = -np.asarray(var), -np.asarray(es), np.asarray(r)
    return -((y <= v) * (v - y)) / (a * e) + v / e + np.log(-e) - 1


def nw_var(d, lags=None):
    d = np.asarray(d) - np.mean(d)
    T = len(d)
    lags = int(np.floor(4 * (T / 100) ** (2 / 9))) if lags is None else lags
    v = d @ d / T
    for l in range(1, lags + 1):
        v += 2 * (1 - l / (lags + 1)) * (d[l:] @ d[:-l]) / T
    return v, lags


def dm_test(l1, l2):
    """Diebold-Mariano: d = l1 - l2; t = mean(d)/sqrt(HAC var/T); d < 0 => modelul 1 mai bun."""
    d = np.asarray(l1) - np.asarray(l2)
    v, lags = nw_var(d)
    t = d.mean() / np.sqrt(v / len(d))
    return dict(mean=d.mean(), t=t, p=2 * stats.norm.sf(abs(t)), lags=lags)


def mcs(losses, B=999, block=20, alpha=0.10, seed=0):
    """Setul de modele de incredere (Hansen, Lunde & Nason, 2011), statistica T_max, bootstrap pe blocuri mobile.
    losses: DataFrame T x m. Intoarce valorile p MCS pentru fiecare model."""
    rng = np.random.default_rng(seed)
    L = losses.values
    T, m = L.shape
    nb = int(np.ceil(T / block))
    idx = [(rng.integers(0, T - block, nb)[:, None] + np.arange(block)).ravel()[:T] for _ in range(B)]
    alive = list(range(m))
    pvals = {}
    pmax = 0.0
    while len(alive) > 1:
        Lm = L[:, alive]
        dbar = Lm.mean(0) - Lm.mean()                       # d_i. = L_i - media modelelor ramase
        boot = np.array([Lm[ix].mean(0) - Lm[ix].mean() for ix in idx])
        se = np.sqrt(((boot - dbar) ** 2).mean(0))
        t = dbar / se
        tmax = t.max()
        tb = ((boot - dbar) / se).max(1)
        p = float((tb >= tmax).mean())
        pmax = max(pmax, p)
        worst = alive[int(np.argmax(t))]
        pvals[losses.columns[worst]] = pmax
        alive.remove(worst)
    pvals[losses.columns[alive[0]]] = 1.0
    return {k: pvals[k] for k in losses.columns}, [c for c in losses.columns if pvals[c] >= alpha]


def comparative_backtest(s_int, s_std, level=0.05):
    """Backtest comparativ (Nolde & Ziegel, 2017) cu scoruri FZ0: d = S(intern) - S(standard).
    Rosu: se respinge H0-, 'intern cel putin la fel de bun' (d > 0 semnificativ);
    verde: se respinge H0+, 'intern cel mult la fel de bun' (d < 0 semnificativ); altfel galben."""
    t = dm_test(s_int, s_std)['t']
    z = stats.norm.ppf(1 - level)
    zone = 'red' if t > z else ('green' if t < -z else 'yellow')
    return dict(t=t, zone=zone)


def holm(p):
    """Valori p ajustate Holm (FWER)."""
    p = np.asarray(p, float)
    o = np.argsort(p)
    m = len(p)
    adj = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[o]), 1)
    out = np.empty(m)
    out[o] = adj
    return out


def bh(p, q=0.05):
    """Benjamini-Hochberg: masca respingerilor la FDR q."""
    p = np.asarray(p, float)
    m = len(p)
    o = np.argsort(p)
    ok = p[o] <= q * np.arange(1, m + 1) / m
    k = np.max(np.nonzero(ok)[0]) + 1 if ok.any() else 0
    rej = np.zeros(m, bool)
    rej[o[:k]] = True
    return rej


# =============================================================================
# 4. PUTEREA TESTULUI KUPIEC
# =============================================================================
def kupiec_region(T, p0=0.01, level=0.05):
    """Numerele de depasiri x pentru care LR_uc respinge la nivelul dat."""
    xs = np.arange(0, T + 1)
    ph = xs / T
    with np.errstate(divide='ignore', invalid='ignore'):
        ll1 = np.where(xs > 0, xs * np.log(np.where(ph > 0, ph, 1)), 0) + \
            np.where(xs < T, (T - xs) * np.log(np.where(ph < 1, 1 - ph, 1)), 0)
    ll0 = xs * np.log(p0) + (T - xs) * np.log(1 - p0)
    lr = -2 * (ll0 - ll1)
    return xs[lr > stats.chi2.ppf(1 - level, 1)]


def kupiec_power(T, p1, p0=0.01, level=0.05):
    rej = kupiec_region(T, p0, level)
    return float(stats.binom.pmf(rej, T, p1).sum()), rej


def kupiec_mde(T, p0=0.01, power=0.80, level=0.05):
    """Cea mai mica rata adevarata > p0 detectata cu puterea ceruta."""
    for p1 in np.arange(p0 + 0.0005, 0.2, 0.0005):
        if kupiec_power(T, p1, p0, level)[0] >= power:
            return float(p1)
    return np.nan


# =============================================================================
# 5. RUPTURA CU DATA NECUNOSCUTA
# =============================================================================
def sup_wald_break(y, trim=0.15, nsim=2000, seed=0, hac=True):
    """Test sup-Wald (Andrews, 1993) pentru o ruptura in media lui y la o data necunoscuta, erori HAC;
    valoarea p prin simularea limitei sup_pi B(pi)^2/(pi(1-pi)) (punte browniana)."""
    y = np.asarray(y, float)
    T = len(y)
    lo, hi = int(np.floor(trim * T)), int(np.ceil((1 - trim) * T))
    best = (-np.inf, None)
    v, _ = nw_var(y - y.mean()) if hac else (np.var(y), 0)
    cs = np.cumsum(y)
    ybar = y.mean()
    for k in range(lo, hi):
        m1, m2 = cs[k - 1] / k, (cs[-1] - cs[k - 1]) / (T - k)
        w = (m2 - m1) ** 2 / (v * (1 / k + 1 / (T - k)))
        if w > best[0]:
            best = (w, k)
    rng = np.random.default_rng(seed)
    n = 2000
    grid = np.arange(int(trim * n), int((1 - trim) * n))
    sims = []
    for _ in range(nsim):
        W = np.cumsum(rng.standard_normal(n)) / np.sqrt(n)
        pi = grid / n
        Bb = W[grid - 1] - pi * W[-1]
        sims.append(np.max(Bb ** 2 / (pi * (1 - pi))))
    sims = np.array(sims)
    return dict(stat=best[0], k=best[1], p=float((sims >= best[0]).mean()), cv5=float(np.percentile(sims, 95)))


# =============================================================================
# 6. EVALUAREA FORMALA A CELOR PATRU MODELE (studiul de caz)
# =============================================================================
def formal_eval(fc, models=('HS', 'Normal', 'GARCH-t', 'FHS'), a=0.01, a_es=0.025, standard='HS'):
    """DQ la VaR 1%; FZ0 la nivelul 2.5% pentru perechea (VaR 2.5%, ES 2.5%); DM pe perechi; MCS; backtest comparativ."""
    out = {'dq': {}, 'fz0': {}, 'dm': {}, 'cb': {}}
    S = {}
    for m in models:
        out['dq'][m] = dq_test(fc['L'], fc[m], a)
        S[m] = fz0(fc['r'], fc[m + '_V25'], fc[m + '_ES'], a_es)
        out['fz0'][m] = float(np.mean(S[m]))
    for i, m1 in enumerate(models):
        for m2 in models[i + 1:]:
            out['dm'][f'{m1}|{m2}'] = dm_test(S[m1], S[m2])
    p, keep = mcs(pd.DataFrame(S))
    out['mcs_p'], out['mcs_set'] = p, keep
    for m in models:
        if m != standard:
            out['cb'][m] = comparative_backtest(S[m], S[standard])
    out['T'] = len(fc)
    return out

## Case-study functions

In [ ]:
ALPHA = 0.01          # VaR 1%
ALPHA_ES = 0.025      # ES 2.5%
WINDOW = 1000         # zile in fereastra de estimare
REFIT = 21            # reestimare GARCH la fiecare 21 de zile
HS_WINDOW = 500       # fereastra pentru HS si pentru distributia Normala
MODELS = ['HS', 'Normal', 'GARCH-t', 'FHS']


# -----------------------------------------------------------------------------
# 1. FAPTE STILIZATE
# -----------------------------------------------------------------------------
def ljung_box(x, m=10):
    """Q(m) = T(T+2) sum rho_k^2/(T-k) si p-valoarea chi2(m)."""
    x = np.asarray(x) - np.mean(x)
    T = len(x)
    rho = np.array([np.sum(x[k:] * x[:-k]) for k in range(1, m + 1)]) / np.sum(x ** 2)
    q = T * (T + 2) * np.sum(rho ** 2 / (T - np.arange(1, m + 1)))
    return q, stats.chi2.sf(q, m)


def acf(x, nlags):
    x = np.asarray(x) - np.mean(x)
    d = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k]) / d for k in range(1, nlags + 1)])


def stylised_facts(r):
    """Rezumatul faptelor stilizate pentru randamentele r (in %)."""
    jb, jbp = stats.jarque_bera(r)
    q_r, q_rp = ljung_box(r)
    q_r2, q_r2p = ljung_box(r ** 2)
    z = (r - r.mean()) / r.std()
    return dict(N=len(r), start=str(r.index[0].date()), end=str(r.index[-1].date()),
                mean=r.mean(), sd=r.std(), skew=stats.skew(r), kurt=stats.kurtosis(r),
                min=r.min(), min_date=str(r.idxmin().date()), max=r.max(),
                jb=jb, jb_p=jbp, q_r=q_r, q_r_p=q_rp, q_r2=q_r2, q_r2_p=q_r2p,
                acf1_r=acf(r, 1)[0], acf1_abs=acf(np.abs(r), 1)[0],
                n4=int(np.sum(np.abs(z) > 4)), n4_normal=len(r) * 2 * stats.norm.sf(4))


def kurtosis_boot_ci(r, B=2000, block=20, seed=0):
    """CI 95% pentru excesul de kurtosis prin bootstrap pe blocuri mobile (pastreaza gruparea volatilitatii)."""
    rng = np.random.default_rng(seed)
    x = np.asarray(r)
    T = len(x)
    nb = int(np.ceil(T / block))
    ks = []
    for _ in range(B):
        idx = (rng.integers(0, T - block, nb)[:, None] + np.arange(block)).ravel()[:T]
        ks.append(stats.kurtosis(x[idx]))
    return np.percentile(ks, [2.5, 97.5])


# -----------------------------------------------------------------------------
# 2. GARCH(1,1)-t
# -----------------------------------------------------------------------------
def garch_t_fit(r, last_obs=None):
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t', rescale=False)
    return am.fit(disp='off', last_obs=last_obs)


def garch_summary(res):
    """Parametri, erori standard robuste, persistenta si timpul de injumatatire."""
    p, se = res.params, res.std_err
    pers = p['alpha[1]'] + p['beta[1]']
    return dict(mu=p['mu'], omega=p['omega'], alpha=p['alpha[1]'], beta=p['beta[1]'], nu=p['nu'],
                se_alpha=se['alpha[1]'], se_beta=se['beta[1]'], se_nu=se['nu'],
                pers=pers, half_life=np.log(0.5) / np.log(pers),
                uncond_vol=np.sqrt(252 * p['omega'] / (1 - pers)))


def t_std_q(nu, a):
    """Cuantila de ordin a a distributiei Student-t standardizate (varianta 1)."""
    return stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu)


def t_std_es(nu, a):
    """E[Z | Z <= q_a] pentru Student-t standardizata (negativ)."""
    q = stats.t.ppf(a, nu)
    return -(stats.t.pdf(q, nu) / a) * (nu + q ** 2) / (nu - 1) * np.sqrt((nu - 2) / nu)


# -----------------------------------------------------------------------------
# 3. PROGNOZE VaR 1% PENTRU ZIUA URMATOARE (fereastra mobila)
# -----------------------------------------------------------------------------
def rolling_var(r, eval_from, window=WINDOW, refit=REFIT, a=ALPHA, a_es=ALPHA_ES):
    """Prognoze VaR (nivel a) si ES (nivel a_es) pentru fiecare zi t >= eval_from, cu informatia pana la t-1.
    Intoarce un DataFrame cu pierderea L_t si VaR/ES pentru HS, Normal, GARCH-t si FHS."""
    r = r.dropna()
    idx = r.index
    start = idx.searchsorted(pd.Timestamp(eval_from))
    start = max(start, window)
    rows = []
    for b0 in range(start, len(r), refit):
        b1 = min(b0 + refit, len(r))
        est = r.iloc[b0 - window:b0]
        am = arch_model(r.iloc[b0 - window:b1], mean='Constant', vol='GARCH', p=1, q=1, dist='t', rescale=False)
        res = am.fit(disp='off', last_obs=window)
        fx = am.fix(res.params)
        sig = fx.conditional_volatility          # sigma_t foloseste informatia pana la t-1
        mu = res.params['mu']
        nu = res.params['nu']
        z = ((est - mu) / sig.iloc[:window]).values   # reziduuri standardizate din fereastra
        zq = np.quantile(z, a)
        zq_es = np.quantile(z, a_es)
        zes = z[z <= zq_es].mean()
        for j in range(b0, b1):
            hist = r.iloc[j - HS_WINDOW:j].values
            s = sig.iloc[j - (b0 - window)]
            L_hist = -hist
            q_hs = np.quantile(L_hist, 1 - a)
            q_hs_es = np.quantile(L_hist, 1 - a_es)
            m, sd = hist.mean(), hist.std(ddof=1)
            rows.append(dict(date=idx[j], r=r.iloc[j], L=-r.iloc[j], sigma=s,
                             HS=q_hs, HS_ES=L_hist[L_hist >= q_hs_es].mean(),
                             Normal=-(m + sd * stats.norm.ppf(a)),
                             Normal_ES=-(m - sd * stats.norm.pdf(stats.norm.ppf(a_es)) / a_es),
                             **{'GARCH-t': -(mu + s * t_std_q(nu, a)),
                                'GARCH-t_ES': -(mu + s * t_std_es(nu, a_es))},
                             FHS=-(mu + s * zq), FHS_ES=-(mu + s * zes),
                             # VaR la nivelul ES (2.5%): perechea (VaR, ES) la acelasi nivel, pentru scorul FZ0
                             HS_V25=q_hs_es, Normal_V25=-(m + sd * stats.norm.ppf(a_es)),
                             **{'GARCH-t_V25': -(mu + s * t_std_q(nu, a_es))}, FHS_V25=-(mu + s * zq_es)))
    return pd.DataFrame(rows).set_index('date')


# -----------------------------------------------------------------------------
# 4. BACKTESTING
# -----------------------------------------------------------------------------
def kupiec(hits, p=ALPHA):
    """Testul de acoperire (POF) al lui Kupiec: LR_uc ~ chi2(1)."""
    hits = np.asarray(hits, int)
    T, x = len(hits), int(hits.sum())
    ph = x / T
    ll0 = (T - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (T - x) * np.log(1 - ph) + x * np.log(ph) if 0 < x < T else 0.0
    lr = -2 * (ll0 - ll1)
    return dict(T=T, x=x, rate=ph, LR=lr, p=stats.chi2.sf(lr, 1))


def christoffersen(hits, p=ALPHA):
    """Testul de independenta (lant Markov de ordinul 1) si testul combinat LR_cc = LR_uc + LR_ind."""
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    n00, n01 = np.sum((a == 0) & (b == 0)), np.sum((a == 0) & (b == 1))
    n10, n11 = np.sum((a == 1) & (b == 0)), np.sum((a == 1) & (b == 1))

    def xlogy(n, q):
        return n * np.log(q) if n > 0 else 0.0
    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l1 = xlogy(n00, 1 - pi01) + xlogy(n01, pi01) + xlogy(n10, 1 - pi11) + xlogy(n11, pi11)
    l0 = xlogy(n00 + n10, 1 - pi) + xlogy(n01 + n11, pi)
    lr_ind = -2 * (l0 - l1)
    lr_uc = kupiec(h, p)['LR']
    return dict(n11=int(n11), pi01=pi01, pi11=pi11, LR_ind=lr_ind, p_ind=stats.chi2.sf(lr_ind, 1),
                LR_cc=lr_uc + lr_ind, p_cc=stats.chi2.sf(lr_uc + lr_ind, 2))


def traffic_light(x, T=250, p=ALPHA):
    """Zona Basel pentru x depasiri in T zile: verde (<5), galben (5-9), rosu (>=10) la T=250, p=1%."""
    cum = stats.binom.cdf(x, T, p)
    return 'green' if cum < 0.95 else ('yellow' if cum < 0.9999 else 'red')


def backtest_table(fc, models=MODELS, a=ALPHA):
    out = {}
    for m in models:
        hits = (fc['L'] > fc[m]).astype(int)
        k, c = kupiec(hits, a), christoffersen(hits, a)
        roll = hits.rolling(250).sum().dropna()
        zones = roll.apply(lambda x: traffic_light(int(x)))
        out[m] = dict(T=k['T'], x=k['x'], rate=k['rate'], LR_uc=k['LR'], p_uc=k['p'],
                      LR_ind=c['LR_ind'], p_ind=c['p_ind'], p_cc=c['p_cc'], n11=c['n11'],
                      share_red=float((zones == 'red').mean()), share_yellow=float((zones == 'yellow').mean()),
                      mean_var=float(fc[m].mean()))
    return out


def binom_band(T, p=ALPHA, level=0.95):
    """Intervalul binomial central pentru rata depasirilor, sub H0: rata = p."""
    lo, hi = stats.binom.ppf([(1 - level) / 2, 1 - (1 - level) / 2], T, p)
    return lo / T, hi / T

## Charts (style and helpers)

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
Teal     = '#17A2B8'
MODEL_COL = {'HS': MainBlue, 'Normal': Orange, 'GARCH-t': Forest, 'FHS': IDAred}

import tempfile
HERE = tempfile.gettempdir()   # intermediate tables (forecast CSV) go to a temporary folder
CHART_DIR = '.'
EVAL_FROM = '2005-01-01'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent."""
    os.makedirs(CHART_DIR, exist_ok=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, axes, ncol=3):
    """O singura legenda sub o figura cu mai multe panouri."""
    h, l = [], []
    for ax in axes:
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l:
                h.append(hh)
                l.append(ll)
    fig.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False)


# =============================================================================
# DATE: randamente log zilnice ale BET, in %
# =============================================================================
r = 100 * log_returns('bet')
RES = {}


# =============================================================================
# 1. FAPTE STILIZATE
# =============================================================================
def part_facts():
    sf = stylised_facts(r)
    h = hill_ci(r.values)                     # indicele de coada (kurtosis-ul nu este consistent daca alpha < 4)
    sf['hill'] = {k: v for k, v in h.items() if k != 'draws'}
    sf['ppy'] = len(r) / ((r.index[-1] - r.index[0]).days / 365.25)
    sf['ann_vol'] = r.std() * np.sqrt(sf['ppy'])
    RES['facts'] = sf

    z = np.sort(((r - r.mean()) / r.std()).values)
    p = (np.arange(1, len(z) + 1) - 0.5) / len(z)
    zn = stats.norm.ppf(p)
    lags = np.arange(1, 51)
    a_r, a_abs = acf(r.values, 50), acf(np.abs(r.values), 50)
    band = 1.96 / np.sqrt(len(r))
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
    ax = axes[0]
    ax.scatter(zn, z, s=4, color=MainBlue, label='BET standardised daily log returns')
    ax.plot([-5, 5], [-5, 5], color=Gray, lw=0.8, ls='--', label='Normal distribution (45-degree line)')
    ax.set_xlabel('Normal quantile')
    ax.set_ylabel('Empirical quantile')
    ax.set_title('Quantile-quantile plot', fontsize=9, loc='left')
    ax = axes[1]
    ax.bar(lags - 0.2, a_r, width=0.4, color=MainBlue, label='ACF of returns $r_t$')
    ax.bar(lags + 0.2, a_abs, width=0.4, color=IDAred, label='ACF of absolute returns $|r_t|$')
    ax.axhspan(-band, band, color=Gray, alpha=0.25, lw=0, label='95% band under i.i.d.')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_xlabel('Lag (trading days)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title('Autocorrelation function', fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend_bottom(fig, axes, ncol=2)
    save_fig('ch19_bet_facts')


# =============================================================================
# 2. GARCH(1,1)-t PE TOATA SELECTIA
# =============================================================================
def part_garch():
    res = garch_t_fit(r)
    g = garch_summary(res)
    RES['garch'] = g
    RES['garch']['profile'] = profile_ci(r.values, res)   # CI profil pentru alpha + beta, restrans la < 1
    sig = res.conditional_volatility
    fig, ax = plt.subplots(figsize=(9.6, 3.2))
    ax.plot(r.index, r.values, color=MainBlue, lw=0.4, label='BET daily log return (%)')
    ax.plot(sig.index, 2 * sig.values, color=IDAred, lw=0.7, label=r'$\pm 2\hat\sigma_t$, GARCH(1,1)-t')
    ax.plot(sig.index, -2 * sig.values, color=IDAred, lw=0.7)
    ax.axhline(0, color=Gray, lw=0.4)
    ax.set_xlim(r.index[0], r.index[-1])
    ax.set_ylabel('%')
    legend_outside_bottom(ax, ncol=2, y=-0.14)
    save_fig('ch19_bet_garch')
    s = sig.copy()
    RES['garch']['sigma_last'] = float(s.iloc[-1])
    RES['garch']['sigma_max'] = float(s.max())
    RES['garch']['sigma_max_date'] = str(s.idxmax().date())
    return res


# =============================================================================
# 3-4. VaR 1% PENTRU ZIUA URMATOARE SI BACKTESTING
# =============================================================================
def part_var(full_res):
    fc = rolling_var(r, EVAL_FROM)
    fc.to_csv(os.path.join(HERE, 'ch19_forecasts_bet.csv'))
    bt = backtest_table(fc)
    RES['backtest'] = bt
    RES['eval'] = dict(start=str(fc.index[0].date()), end=str(fc.index[-1].date()), T=len(fc))
    lo, hi = binom_band(len(fc))
    RES['eval']['band'] = [lo, hi]
    RES['es'] = {m: float(fc[m + '_ES'].mean()) for m in MODELS}
    RES['formal'] = formal_eval(fc)
    pw = {f'{T}_{p1}': kupiec_power(T, p1)[0] for T in (250, 1000, len(fc)) for p1 in (0.013, 0.015, 0.02)}
    RES['power'] = dict(pw=pw, mde={str(T): kupiec_mde(T) for T in (250, 1000, len(fc))}, T=len(fc),
                        rej250=[int(x) for x in kupiec_power(250, 0.02)[1][:12]])
    fig, ax = plt.subplots(figsize=(7.6, 3.0))
    grid = np.arange(0.002, 0.0405, 0.0005)
    for T, c in [(250, Orange), (1000, MainBlue), (len(fc), IDAred)]:
        ax.plot(100 * grid, [kupiec_power(T, p1)[0] for p1 in grid], color=c, lw=1.2, label=f'T = {T} days')
    ax.axhline(0.8, color=Gray, lw=0.6, ls='--')
    ax.axhline(0.05, color=Gray, lw=0.6, ls=':')
    ax.axvline(1.0, color='black', lw=0.6, ls='--', label='Null: 1%')
    ax.set_xlabel('True breach rate of the VaR 1% model (%)')
    ax.set_ylabel('Rejection probability of\nthe 5% Kupiec test')
    legend_outside_bottom(ax, ncol=4, y=-0.2)
    save_fig('ch19_kupiec_power')

    # grafic: pierderi si VaR 1% (2020-2026)
    d = fc.loc['2020-01-01':]
    fig, ax = plt.subplots(figsize=(9.6, 3.3))
    ax.bar(d.index, d['L'].clip(lower=0), width=1.5, color=Teal, alpha=0.7, label='Daily loss (gains set to 0)')
    ax.plot(d.index, d['Normal'], color=Orange, lw=0.9, label='Normal VaR 1% (500 days)')
    ax.plot(d.index, d['FHS'], color=IDAred, lw=0.9, label='FHS VaR 1% (GARCH-t filter)')
    exc = d['L'] > d['FHS']
    ax.scatter(d.index[exc], d['L'][exc], s=12, color='black', zorder=3, label='Loss above FHS VaR 1%')
    ax.set_ylabel('Loss (% of position)')
    legend_outside_bottom(ax, ncol=2, y=-0.14)
    save_fig('ch19_bet_var')
    RES['var_2020'] = dict(n=len(d), x_fhs=int(exc.sum()), x_normal=int((d['L'] > d['Normal']).sum()),
                          fhs_max=float(d['FHS'].max()), fhs_max_date=str(d['FHS'].idxmax().date()),
                          fhs_last=float(fc['FHS'].iloc[-1]), normal_last=float(fc['Normal'].iloc[-1]))

    # grafic: ratele depasirilor
    fig, ax = plt.subplots(figsize=(7.6, 3.2))
    x = np.arange(len(MODELS))
    rates = [100 * bt[m]['rate'] for m in MODELS]
    ax.bar(x, rates, width=0.55, color=[MODEL_COL[m] for m in MODELS])
    ax.axhspan(100 * lo, 100 * hi, color=Gray, alpha=0.25, lw=0, label='95% binomial band if the rate is 1%')
    ax.axhline(1.0, color='black', lw=0.8, ls='--', label='Target: 1%')
    for i, v in enumerate(rates):
        ax.text(i, v + 0.04, f'{v:.2f}%', ha='center', va='bottom', fontsize=8, color='black')
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS)
    ax.set_ylabel('Share of days with loss > VaR 1% (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.16)
    save_fig('ch19_breach_rates')

    # grafic: depasiri pe 250 de zile si zonele Basel
    fig, ax = plt.subplots(figsize=(9.6, 3.3))
    ax.axhspan(-0.5, 4.5, color=Forest, alpha=0.08, lw=0, label='Green zone (0-4)')
    ax.axhspan(4.5, 9.5, color=Amber, alpha=0.15, lw=0, label='Yellow zone (5-9)')
    top = 0
    for m in MODELS:
        roll = (fc['L'] > fc[m]).astype(int).rolling(250).sum()
        top = max(top, roll.max())
        ax.plot(roll.index, roll.values, color=MODEL_COL[m], lw=0.9, label=m)
    ax.axhspan(9.5, top + 2, color=IDAred, alpha=0.08, lw=0, label='Red zone (10+)')
    ax.set_ylim(-0.5, top + 2)
    ax.set_xlim(fc.index[0], fc.index[-1])
    ax.set_ylabel('Breaches in the last 250 days')
    legend_outside_bottom(ax, ncol=4, y=-0.14)
    save_fig('ch19_rolling_breaches')

    # capcana: informatie din viitor (parametri estimati pe toata selectia)
    p = full_res.params
    sig_full = full_res.conditional_volatility.loc[fc.index]
    var_full = -(p['mu'] + sig_full * t_std_q(p['nu'], ALPHA))
    hs_full = np.quantile(fc['L'], 1 - ALPHA)
    k_full = kupiec((fc['L'] > var_full).astype(int))
    k_hsfull = kupiec((fc['L'] > hs_full).astype(int))
    RES['lookahead'] = dict(garch_full_rate=k_full['rate'], garch_full_p=k_full['p'], garch_full_x=k_full['x'],
                            hs_full_rate=k_hsfull['rate'], hs_full_x=k_hsfull['x'], hs_full_var=float(hs_full))
    fig, ax = plt.subplots(figsize=(7.6, 3.0))
    lab = ['HS', 'GARCH-t']
    oos = [100 * bt['HS']['rate'], 100 * bt['GARCH-t']['rate']]
    ins = [100 * k_hsfull['rate'], 100 * k_full['rate']]
    x = np.arange(2)
    ax.bar(x - 0.18, oos, 0.34, color=MainBlue, label='Rolling estimation (only past data)')
    ax.bar(x + 0.18, ins, 0.34, color=Amber, label='Parameters from the full sample (look-ahead)')
    ax.axhline(1.0, color='black', lw=0.8, ls='--', label='Target: 1%')
    for i in range(2):
        ax.text(i - 0.18, oos[i] + 0.03, f'{oos[i]:.2f}%', ha='center', va='bottom', fontsize=8, color='black')
        ax.text(i + 0.18, ins[i] + 0.03, f'{ins[i]:.2f}%', ha='center', va='bottom', fontsize=8, color='black')
    ax.set_xticks(x)
    ax.set_xticklabels(lab)
    ax.set_ylabel('Breach rate of VaR 1% (%)')
    legend_outside_bottom(ax, ncol=3, y=-0.12)
    save_fig('ch19_lookahead')

## Seminar functions

In [ ]:
HERE = '.'
FTSE = '2020-09-21'      # BVB inclusa in indicii FTSE Russell pentru piete emergente secundare
RES = {}

bet = 100 * log_returns('bet')
btc = 100 * log_returns('btc')


def half_life_ci(res):
    """Timpul de injumatatire ln(0.5)/ln(alpha+beta) cu CI 95% prin metoda delta (covarianta robusta)."""
    p = res.params
    cov = res.param_cov.loc[['alpha[1]', 'beta[1]'], ['alpha[1]', 'beta[1]']].values
    pers = p['alpha[1]'] + p['beta[1]']
    se_pers = np.sqrt(np.ones(2) @ cov @ np.ones(2))
    hl = np.log(0.5) / np.log(pers)
    d = -np.log(0.5) / (pers * np.log(pers) ** 2)
    se_hl = abs(d) * se_pers
    lo = pers - 1.96 * se_pers
    return dict(pers=pers, se_pers=se_pers, pers_lo=lo, pers_hi=pers + 1.96 * se_pers,
                hl=hl, se_hl=se_hl, hl_at_lo=np.log(0.5) / np.log(lo))


def robust_q(x, m=10):
    """Portmanteau robust la heteroscedasticitate conditionata (Lobato, Nankervis & Savin, 2001):
    Q* = T sum_k rho_k^2 / tau_k, tau_k = mean(x_t^2 x_{t-k}^2) / gamma_0^2, ~ chi2(m) sub H0 (necorelare)."""
    x = np.asarray(x) - np.mean(x)
    T = len(x)
    g0 = np.mean(x ** 2)
    q = 0.0
    for k in range(1, m + 1):
        rho = np.mean(x[k:] * x[:-k]) / g0
        tau = np.mean((x[k:] * x[:-k]) ** 2) / g0 ** 2
        q += T * rho ** 2 / tau
    return q, stats.chi2.sf(q, m)


def part_facts():
    for k, r in [('bet', bet), ('btc', btc)]:
        sf = stylised_facts(r)
        h = hill_ci(r.values, seed=1)
        sf['hill'] = {kk: v for kk, v in h.items() if kk != 'draws'}
        sf['q_robust'], sf['q_robust_p'] = robust_q(r.values)
        sf['q_r_raw'], sf['q_r_raw_p'] = ljung_box(r.values)
        RES[f'facts_{k}'] = sf
    d = RES['facts_btc']['hill']['alpha'] - RES['facts_bet']['hill']['alpha']
    se = np.sqrt(RES['facts_btc']['hill']['se_boot'] ** 2 + RES['facts_bet']['hill']['se_boot'] ** 2)
    RES['hill_diff'] = dict(d=d, se=se, p=2 * stats.norm.sf(abs(d / se)))


def part_garch():
    res = garch_t_fit(bet)
    g = garch_summary(res)
    g.update(half_life_ci(res))
    g['loglik'] = res.loglikelihood
    g['profile'] = profile_ci(bet.values, res)
    RES['garch_bet'] = g
    gjr = arch_model(bet, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='t', rescale=False).fit(disp='off')
    lr = 2 * (gjr.loglikelihood - res.loglikelihood)
    RES['gjr_bet'] = dict(alpha=gjr.params['alpha[1]'], gamma=gjr.params['gamma[1]'], beta=gjr.params['beta[1]'],
                          se_gamma=gjr.std_err['gamma[1]'], t_gamma=gjr.tvalues['gamma[1]'],
                          LR=lr, p=stats.chi2.sf(lr, 1), aic_garch=res.aic, aic_gjr=gjr.aic,
                          bic_garch=res.bic, bic_gjr=gjr.bic)


def part_var_btc():
    fc = rolling_var(btc, '2019-01-01')
    RES['bt_btc'] = backtest_table(fc)
    RES['formal_btc'] = formal_eval(fc)
    fc.to_csv(os.path.join(HERE, 'ch19_forecasts_btc.csv'))
    RES['eval_btc'] = dict(start=str(fc.index[0].date()), end=str(fc.index[-1].date()), T=len(fc))
    d = fc.loc['2022-01-01':]
    fig, ax = plt.subplots(figsize=(9.6, 3.3))
    ax.bar(d.index, d['L'].clip(lower=0), width=1.2, color=Teal, alpha=0.7, label='Daily loss (gains set to 0)')
    ax.plot(d.index, d['HS'], color=MainBlue, lw=0.9, label='HS VaR 1% (500 days)')
    ax.plot(d.index, d['FHS'], color=IDAred, lw=0.9, label='FHS VaR 1% (GARCH-t filter)')
    exc = d['L'] > d['FHS']
    ax.scatter(d.index[exc], d['L'][exc], s=12, color='black', zorder=3, label='Loss above FHS VaR 1%')
    ax.set_ylabel('Loss (% of position)')
    legend_outside_bottom(ax, ncol=2, y=-0.14)
    save_fig('ch19_sem_btc_var')


def part_join():
    """Corelatia Bitcoin-SPY: (gresit) randamente pe calendare proprii, apoi join; (corect) join pe preturi."""
    spy = asset_price('SPY')
    b = asset_price('BTC')
    r_spy = np.log(spy).diff().dropna()
    r_btc = np.log(b).diff().dropna()
    wrong = pd.concat([r_btc, r_spy], axis=1, join='inner').dropna()
    p = pd.concat([b, spy], axis=1, join='inner').dropna()
    right = np.log(p).diff().dropna()
    out = {}
    for lab, a, z in [('2015_2019', '2015-01-01', '2019-12-31'), ('2020_2026', '2020-01-01', '2026-09-18')]:
        w, rr = wrong.loc[a:z], right.loc[a:z]
        mon_w = w[w.index.dayofweek == 0]
        mon_r = rr[rr.index.dayofweek == 0]
        out[lab] = dict(wrong=w.corr().iloc[0, 1], right=rr.corr().iloc[0, 1],
                        wrong_mon=mon_w.corr().iloc[0, 1], right_mon=mon_r.corr().iloc[0, 1], n=len(rr),
                        sd_btc_wrong=100 * w.iloc[:, 0].std(), sd_btc_right=100 * rr.iloc[:, 0].std(),
                        sd_btc_mon_wrong=100 * mon_w.iloc[:, 0].std(), sd_btc_mon_right=100 * mon_r.iloc[:, 0].std(),
                        cum_btc_wrong=float(np.exp(w.iloc[:, 0].sum()) - 1), cum_btc_right=float(np.exp(rr.iloc[:, 0].sum()) - 1))
    RES['join'] = out
    rw = wrong.loc['2016-01-01':]
    rr = right.loc['2016-01-01':]
    cw = rw.iloc[:, 0].rolling(250).corr(rw.iloc[:, 1]).dropna()
    cr = rr.iloc[:, 0].rolling(250).corr(rr.iloc[:, 1]).dropna()
    fig, ax = plt.subplots(figsize=(9.6, 3.2))
    ax.plot(cr.index, cr.values, color=MainBlue, lw=1.0, label='Join on prices, then returns (correct)')
    ax.plot(cw.index, cw.values, color=Orange, lw=1.0, label='Returns on own calendars, then join (wrong)')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_ylabel('250-day correlation, Bitcoin vs SPY')
    legend_outside_bottom(ax, ncol=2, y=-0.14)
    save_fig('ch19_sem_join')


def part_c():
    """Analiza de referinta pentru C1: BET inainte si dupa reclasificarea FTSE (21.09.2020), control WIG20."""
    pre, post = bet.loc['2014-09-22':FTSE].iloc[:-1], bet.loc[FTSE:]
    out = {}
    hd = {}
    for lab, r in [('pre', pre), ('post', post)]:
        res = garch_t_fit(r)
        g = garch_summary(res)
        h = hill_ci(r.values, seed=5)
        hd[lab] = h['draws']
        out[lab] = dict(N=len(r), start=str(r.index[0].date()), end=str(r.index[-1].date()),
                        vol=r.std() * np.sqrt(252), kurt=stats.kurtosis(r), hill=h['alpha'], hill_lo=h['lo'],
                        hill_hi=h['hi'], k=h['k'], pers=g['pers'], nu=g['nu'], hs_var=float(np.quantile(-r, 0.99)))
    # diferenta indicilor Hill: extrageri independente in cele doua perioade disjuncte
    dd = hd['post'] - hd['pre']
    out['dhill'] = out['post']['hill'] - out['pre']['hill']
    out['dhill_ci'] = list(np.percentile(dd, [2.5, 97.5]))
    # sensibilitatea kurtosis-ului (necontrolat cand alpha < 4) la cele mai mari zile
    top = pre.abs().nlargest(2).index
    out['pre']['top_days'] = [str(d.date()) for d in top]
    out['pre']['top_returns'] = [float(pre.loc[d]) for d in top]
    out['pre']['kurt_ex1'] = stats.kurtosis(pre.drop(top[:1]))
    out['pre']['kurt_ex2'] = stats.kurtosis(pre.drop(top))
    out['dkurt'] = stats.kurtosis(post) - stats.kurtosis(pre)
    # control: WIG20 (neafectat de reclasificarea BVB), diferenta-in-diferente pe indicele Hill
    wp = asset_price('WIG20')
    wp = wp[wp.diff() != 0]                                  # sarbatori completate cu valoarea anterioara
    w = 100 * np.log(wp).diff().dropna()
    wpre, wpost = w.loc['2014-09-22':FTSE].iloc[:-1], w.loc[FTSE:'2026-09-18']
    hw = {lab: hill_ci(r.values, seed=6) for lab, r in [('pre', wpre), ('post', wpost)]}
    out['wig'] = dict(pre=hw['pre']['alpha'], post=hw['post']['alpha'], d=hw['post']['alpha'] - hw['pre']['alpha'])
    did = dd - (hw['post']['draws'] - hw['pre']['draws'])
    out['did'] = out['dhill'] - out['wig']['d']
    out['did_ci'] = list(np.percentile(did, [2.5, 97.5]))
    # ruptura cu data necunoscuta in log|r| (toate momentele finite), 2014-2026
    y = np.log(np.abs(bet.loc['2014-09-22':'2026-09-18']))
    sw = sup_wald_break(y.values)
    out['supwald'] = dict(stat=sw['stat'], p=sw['p'], cv5=sw['cv5'], date=str(y.index[sw['k']].date()))
    # date placebo: aceeasi analiza Hill la +/- un an
    out['placebo'] = {}
    for lab, d0 in [('m1', '2019-09-23'), ('p1', '2021-09-20')]:
        a1, a2 = bet.loc['2014-09-22':d0].iloc[:-1], bet.loc[d0:'2026-09-18']
        h1, h2 = hill_ci(a1.values, seed=7), hill_ci(a2.values, seed=8)
        dpl = h2['draws'] - h1['draws']
        out['placebo'][lab] = dict(date=d0, d=h2['alpha'] - h1['alpha'], lo=float(np.percentile(dpl, 2.5)),
                                   hi=float(np.percentile(dpl, 97.5)))
    fc = rolling_var(bet, '2014-09-22')
    for lab, a, z in [('pre', '2014-09-22', '2020-09-18'), ('post', FTSE, '2026-09-18')]:
        d = fc.loc[a:z]
        out[lab]['fhs_rate'] = float((d['L'] > d['FHS']).mean())
        out[lab]['hs_rate'] = float((d['L'] > d['HS']).mean())
        out[lab]['T_fc'] = len(d)
    RES['c1'] = out

## Analysis

In [ ]:
part_facts()
part_garch()
part_var_btc()
part_join()
part_c()
print(json.dumps({k: RES[k] for k in ['garch_bet', 'gjr_bet', 'join', 'c1']}, indent=1, default=float))

![ch19_sem_btc_var](./ch19_sem_btc_var.png)

Output: `ch19_sem_btc_var.pdf`

![ch19_sem_join](./ch19_sem_join.png)

Output: `ch19_sem_join.pdf`